# IFEval: the protocol grid — solo and panel, with and without correction

[IFEval](https://arxiv.org/abs/2311.07911) contains 541 instruction-following prompts with
deterministic checks for requirements such
as word counts, required sections, and forbidden punctuation. Grading uses the vendored official
verifier and makes no grading-model calls.

This notebook runs the SAME benchmark (`ifeval`) across a 2x2 protocol grid, varying exactly one
dimension at a time:

|                | solo                    | panel                                     |
|----------------|-------------------------|-------------------------------------------|
| **no loop**    | plain `sf.Model`        | `sf.Fusion` (drafts blended once)         |
| **corrective** | `sf.SelfCorrective`     | `sf.CorrectiveLoop` (drafts checked, best |
|                | (self-coached retries)  | passing draft submitted verbatim)         |

where `sf.CorrectiveLoop` is the protocol from
[this paper](https://openreview.net/pdf?id=XSIYfTm2h7) 

![The IFEval protocol grid: solo vs panel, no loop vs corrective](assets/ifeval-protocol-grid.png)


## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105, Scoreboard :9106, and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

sf.connect()

## Define models and fusion

In [ ]:
ANSWER_PROMPT = (
    "Answer the request accurately and completely. "
    "Follow every instruction and formatting constraint in the request."
)

PARAMS = {"max_tokens": 8192, "temperature": 0.0}

ministral = sf.Model(
    model="openrouter/mistralai/ministral-3b-2512",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)
phi = sf.Model(
    model="openrouter/microsoft/phi-4",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)

In [ ]:
SYNTHESIS_PROMPT = (
    "Produce one final answer to the original request from the panel drafts. "
    "Preserve every instruction and formatting constraint."
)

deepseek = sf.Model(
    model="openrouter/deepseek/deepseek-v4-flash",
    prompt=SYNTHESIS_PROMPT,
    params=PARAMS,
)

light_open_source = sf.Fusion(
    members=[ministral, phi], name="light_open_source", synthesizer=deepseek
)

## 1. Solo, no loop — canonical baseline

In [ ]:
canonical_solo = sf.evaluate(phi, benchmark="ifeval", limit=1)
canonical_solo

## 2. Panel, no loop — whole-Fusion synthesis

In [ ]:
canonical_fusion = sf.evaluate(light_open_source, benchmark="ifeval", limit=1)
canonical_fusion

## 3. Solo, corrective — `sf.SelfCorrective`

The same model re-sits the exam up to three times, authoring its own study notes from the
check surface's sanitized feedback between sittings. A first-round pass costs one draft and
one free check — nothing else.

In [ ]:
self_corrective = sf.evaluate(
    sf.SelfCorrective(phi, max_rounds=3),
    benchmark="ifeval",
    limit=1,
)
self_corrective

## 4. Panel, corrective — `sf.CorrectiveLoop`

In [ ]:
corrective_loop = sf.CorrectiveLoop(members=[ministral, phi], judge=deepseek, max_rounds=3)
corrective_loop

In [ ]:
corrective_loop_report = sf.evaluate(
    corrective_loop,
    benchmark="ifeval",
    limit=2,
)
corrective_loop_report

## 5. Send the score to the Scoreboard

Publication takes the evaluated `CandidateResult` and submits the Benchmark's **native
score** exactly as the Engine graded it — fractional or negative values included — and the
Scoreboard stores and ranks it without recalculating. Opt-in so **Run All** never changes
the public Leaderboard.

In [ ]:
PUBLISH_RESULT = False

submission = (
    sf.leaderboards.submit(corrective_loop_report.candidates.only) if PUBLISH_RESULT else None
)
submission